# Try Vikhr for few-shot simplification

In [1]:
import pathlib as pth

base_location  = pth.Path.cwd().parent.parent

vikhr_location  = base_location / "models" / "vikhr" / "Vikhr-7B-instruct_0.4-Q6_K.gguf"

## Load Model

In [2]:
from langchain_community.llms import LlamaCpp


model = LlamaCpp(
            model_path=vikhr_location.as_posix(),
            n_gpu_layers=16,
            n_batch=512,
            temperature=0.8,
            max_tokens=256,
            top_p=5,
            verbose=False,
            n_ctx=8192,
            f16_kv=True,
            repeat_penalty=1.1,
        )

## Set Prompts

Vikhr is Russian-language modeel, based on Qwen. Thus, it is only reasonable to write prompts in Russian.

In [3]:
def format_prompt(user_text: str, level: int) -> str:
    if level == 1:
        prompt = (
            """<|im_start|>system\n"""
            "Ты — помощник для упрощения текста на русском языке. "
            "Тебе дан текст на русском языке, и ты должна предоставить его упрощённую версию на русском языке. "
            "Избегай объединения слишком большого количества деталей в одно предложение; распределяй их по нескольким предложениям, если это необходимо. "
            "Сохраняй важную информацию, такую как имена, национальности и роли. Не удаляй важные детали. "
            "Перефразируй предложения, удаляя причастные и деепричастные обороты. "
            "По возможности заменяй пассивный залог на активный. "
            "Если предложение состоит только из существительного, добавь глагол. "
            "Редкие или малоупотребительные слова заменяй на более распространённые. "
            "Где уместно, удаляй или заменяй иностранные слова. "
            "Неясные фразы заменяй более конкретными, легко понимаемыми словами. "
            "По возможности избегай слов, имеющих паронимы. "
            "Используй только русский язык, английский запрещён.<|im_end|>\n"
            "<|im_start|>user\n"
            "Упрости следующий текст на русском языке: {user_text}<|im_end|>\n"
            "<|im_start|>assistant\nУпрощенный текст: "
        )
        
    elif level == 2:
        prompt = (
            """<|im_start|>system\n"""
            "Ты — помощник для упрощения текста на русском языке. "
            "Тебе дан текст на русском языке, и ты должна предоставить его упрощённую версию на русском языке. "
            "Упрощай сложные или составные предложения, разбивая их на короткие, с длиной не более семи слов. "
            "Убедись, что каждое предложение содержит только одну идею. "
            "Избегай причастных и деепричастных оборотов, отдавай предпочтение активному залогу вместо пассивного. "
            "Сохраняй важную информацию, такую как имена, национальности и роли. Не удаляй важные детали. "
            "Удаляй ненужные иностранные слова (например, названия брендов) и заменяй редкие или длинные слова на более простые и короткие. "
            "Упрощай двойственные по смыслу фразы, используя конкретные и ясные слова. "
            "Удаляй незначительные детали, не добавляющие важного смысла, но оставляй ключевую информацию. "
            "Используй только русский язык, английский запрещён.<|im_end|>\n"
            "<|im_start|>user\n"
            "Упрости следующий текст на русском языке: {user_text}<|im_end|>\n"
            "<|im_start|>assistant\nУпрощенный текст: "
        )

    elif level == 3:
        prompt = (
            """<|im_start|>system\n"""
            "Ты — помощник для упрощения текста на русском языке. "
            "Тебе дан текст на русском языке, и ты должна предоставить его упрощённую версию на русском языке. "
            "Твоя задача — сделать текст максимально простым. "
            "Каждое предложение должно содержать только одну идею и быть длиной не более пяти слов. "
            "Удаляй или заменяй иностранные слова (такие как имена, места или бренды), избегай незначительных деталей. "
            "Исключай числительные и удаляй ненужные подробности. "
            "Используй только именительный и родительный падежи для существительных и только настоящее или прошедшее время для глаголов. "
            "Избегай пассивного залога и инверсии слов. "
            "Редкие или малоупотребительные слова заменяй на более распространённые. "
            "Заменяй сложные фразы на общеупотребительные выражения, клише или идиомы. "
            "Удаляй лишние детали (если это возможно без искажения смысла предложения) и максимально упрощай неясные фразы."
            "Используй только русский язык, английский запрещён.<|im_end|>\n"
            "<|im_start|>user\n"
            "Упрости следующий текст на русском языке: {user_text}<|im_end|>\n"
            "<|im_start|>assistant\nУпрощенный текст: "
        )
    
    return prompt.format(user_text=user_text)


### 1 Level

In [4]:
prompt = format_prompt(user_text="Россиянка Елена Максимова одержала победу в международном конкурсе «Миссис Вселенная».", level=1)

model.invoke(str(prompt))

'1. "Россиянка" означает "россиянка". "Миссис Вселенная" относится к международному конкурсу для замужних женщин. "Победа" - это успех или триумф в соревновании.\n2. Елена Максимова, россиянка, стала победительницей международного конкурса "Миссис Вселенная", победив других участниц и продемонстрировав свои навыки и красоту.'

### 2 Level

In [5]:
prompt = format_prompt(user_text="Россиянка Елена Максимова одержала победу в международном конкурсе «Миссис Вселенная».", level=2)

model.invoke(str(prompt))

'23-летняя российская модель Елена Максимова стала победительницей международного конкурса "Mrs Universe".'

### 3 Level

In [6]:
prompt = format_prompt(user_text="Россиянка Елена Максимова одержала победу в международном конкурсе «Миссис Вселенная».", level=3)

model.invoke(str(prompt))

'\nЕлена Максимова из России стала победительницей конкурса "Миссис Вселенная".'

## Load Data

In [7]:
import pandas as pd

data_location = base_location / "data" / "RuSimpleSentAphasia.csv"
data = pd.read_csv(data_location.as_posix())

data.head()

,source,level 1,level 2,level 3
0,Россиянка Елена Максимова одержала победу в ме...,Россиянка Елена Максимова победила в конкурсе ...,Россиянка победила в конкурсе «Миссис Вселенная».,Россиянка победила в конкурсе «Миссис Вселенная».
1,Представительница России впервые завоевала это...,"В прессе сказали, что участница из России полу...",Участница из России получает этот титул впервые.,Россиянка получает этот титул впервые.
2,"Уточняется, что финал прошел в Софии 4 февраля.",Финал прошел в Болгарии в начале февраля.,Финал был в Болгарии в феврале.,Финал был начале февраля. Он был в в Болгарии.
3,Участие в нем принимали 120 женщин из разных с...,В нем участвовали 120 женщин из разных стран.,В нем участвовали 120 женщин из разных стран.,В нем участвовали женщины из разных стран.
4,«Конкуренция на конкурсе была очень жесткая: р...,«В конкурсе было сложно выиграть. Было много у...,«В конкурсе было сложно выиграть. Было много у...,«В конкурсе было сложно выиграть.


In [8]:
new_data = pd.DataFrame(data["source"].head(200))

In [9]:
new_data = pd.DataFrame(data["source"].head(200))

In [10]:
def apply_generation(text, level) -> str:
    prompt = format_prompt(user_text=text, level=level)
    return model.invoke(str(prompt), stop=["\n\n", " \n\n", ". \n\n"])

In [11]:
for level in [1, 2, 3]:
    column = f"level {level}"
    new_data[column] = new_data["source"].apply(lambda x: apply_generation(x, level))

In [12]:
new_data_location = base_location / "data" / "RuSimpleSentAphasia_200_generated_vikhr.csv"

new_data.to_csv(new_data_location.as_posix(), index=False)

## Let's calculate BERTscore between ground truth and predicted texts

In [13]:
new_data

,source,level 1,level 2,level 3
0,Россиянка Елена Максимова одержала победу в ме...,43-летняя российская женщина Елена Максимова с...,\nРоссиянка Елена Максимова выиграла междунаро...,"\nРоссиянка Елена Максимова выиграла конкурс ""..."
1,Представительница России впервые завоевала это...,"1-ая россиянка выиграла конкурс Мисс Мира, соо...","Россиянин выиграл первый конкурс ""Мистер Мира"".","1-я российская участница победила в конкурсе, ..."
2,"Уточняется, что финал прошел в Софии 4 февраля.",4 февраля в Софии прошел финал.,4 февраля в Софии был финал.,4 февраля финал состоялся в Софии.
3,Участие в нем принимали 120 женщин из разных с...,120 женщин участвовали в мероприятии из стран ...,120 женщин из 11 стран участвовали в мероприятии.,120 девушек из разных стран участвовали - Кита...
4,«Конкуренция на конкурсе была очень жесткая: р...,"\n""Конкурс был очень трудным: было много участ...",400 участников боролись за победу в соревнован...,34 участника соревновались за приз - много тал...
...,...,...,...,...
195,Ранее президент Национального института геофиз...,\nПрезидент Института геофизики и вулканологии...,15 августа мощное землетрясение произошло в Ту...,30 октября мощное землетрясение произошло в Ту...
196,Во время операции врачи извлекли 39 металличес...,30-летний мужчина из Ливана перенес операцию п...,"30-летний ливанец имел в животе ножи, вилки и ...","30-летний ливанец удалил ножи, вилки, ложки и ..."
197,"По словам одного из хирургов, этот пациент поп...","1 хирург сказал, что пациент был доставлен в б...","1 хирург говорит, что больной попал в больницу...","1 хирург сказал, что пациент поступил в больни..."
198,"Выяснилось, что все эти предметы пациент прогл...",14-летний мальчик проглотил множество предмето...,,\nПациент случайно съел все эти предметы за год.


In [14]:
from evaluate import load
import numpy as np

bertscore = load("bertscore")

bert_scores = {}
val_data = data.head(200)


for level in val_data.columns[1:]:
    references = val_data[level].tolist()
    predictions = new_data[level].tolist()

    results = bertscore.compute(predictions=predictions, references=references, lang="ru")
    
    bert_scores[level] = {
        "precision": np.mean(results['precision']),
        "recall": np.mean(results['recall']),
        "f1": np.mean(results['f1'])
    }

/home/z00logist/hse-simplification-for-aphasia/.venv/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [15]:
for level, scores in bert_scores.items():
    print(f"BERTScore for {level}:")
    print(f"Precision: {scores['precision']}")
    print(f"Recall: {scores['recall']}")
    print(f"F1: {scores['f1']}\n")

BERTScore for level 1:
Precision: 0.7198711943626404
Recall: 0.7611918166279793
F1: 0.7393784761428833

BERTScore for level 2:
Precision: 0.7260954862833023
Recall: 0.768082629442215
F1: 0.7457137805223465

BERTScore for level 3:
Precision: 0.7024051651358605
Recall: 0.7459733548760414
F1: 0.7229879215359688

